# Factor 1 — GC Content / GC 含量差異

**Owner / 負責人:** Alex Chen  
**Step:** 03 Feature Weighting → Factor 1  
**Expected runtime / 預計執行時間:** ~5 min first run (host genome download), <30 s cached / 首次約 5 分鐘（需下載宿主基因組），後續快取後 30 秒內完成

---

## Biological rationale / 生物學原理

GC content reflects long-term co-evolution between a phage and its host. When a phage infects a host for many generations, selection pressure tends to align their genomic GC% so that phage genes are translated efficiently by the host's tRNA pool. A smaller `|GC_phage − GC_host|` therefore correlates with higher infection likelihood.

GC 含量反映噬菌體與宿主之間的長期共演化關係。當噬菌體長期感染某一宿主時，天擇壓力會使兩者的 GC% 趨向一致，以確保噬菌體基因能被宿主 tRNA 池高效翻譯。因此，`|GC噬菌體 − GC宿主|` 越小，感染可能性越高。

**Feature definition / 特徵定義:**  
`x = |GC%_phage − GC%_host|` (whole-genome, fraction 0–1)

In [ ]:
# ── Imports ─────────────────────────────────────────────────────────────────
import os
import subprocess
import zipfile
import tempfile
import pathlib
import warnings

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from Bio import SeqIO
from Bio.SeqUtils import gc_fraction

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='muted')
print('Imports OK')

In [ ]:
# ── Config — edit paths here if your layout differs ─────────────────────────
REPO_ROOT = pathlib.Path(__file__).parents[3] if '__file__' in dir() else pathlib.Path().resolve().parents[2]
# ^ resolves to iGEM_Claremont_2026/ regardless of where Jupyter was launched

TRUTH_TABLE_PATH = REPO_ROOT / '02_annotation/outputs/host_proteins/pairs.csv'
# When you receive the updated truth table, replace the path above.
# 收到更新版真值表後，將上方路徑替換即可。

MANIFEST_PATH    = REPO_ROOT / '02_annotation/outputs/host_proteins/manifest.csv'
PHAGE_RUNS_DIR   = REPO_ROOT / '02_annotation/outputs/pharokka_runs'
HOST_GENOMES_DIR = REPO_ROOT / '01_data_ground_truth/outputs/host_genomes/by_organism'
OUT_DIR          = REPO_ROOT / '03_feature_weighting/outputs/per_factor/factor1'

HOST_GENOMES_DIR.mkdir(parents=True, exist_ok=True)
OUT_DIR.mkdir(parents=True, exist_ok=True)

print(f'REPO_ROOT      : {REPO_ROOT}')
print(f'TRUTH_TABLE    : {TRUTH_TABLE_PATH}')
print(f'MANIFEST       : {MANIFEST_PATH}')
print(f'PHAGE_RUNS_DIR : {PHAGE_RUNS_DIR}')
print(f'HOST_GENOMES   : {HOST_GENOMES_DIR}')
print(f'OUT_DIR        : {OUT_DIR}')

In [ ]:
# ── Cell 3: Acquire host genomes via NCBI datasets CLI ──────────────────────
# Idempotent: skips hosts already on disk.
# 冪等操作：若 .fna 已存在則跳過，不重複下載。
#
# Requires: conda install -c conda-forge ncbi-datasets-cli

manifest = pd.read_csv(MANIFEST_PATH)
print(f'Manifest: {len(manifest)} hosts')
display(manifest[['host_id', 'kind', 'assembly', 'organism']])

def fetch_host_genome(host_id: str, gcf: str, out_dir: pathlib.Path) -> pathlib.Path:
    """Download host assembly .fna via datasets CLI. Returns path to .fna file."""
    dest_dir = out_dir / host_id
    dest_dir.mkdir(parents=True, exist_ok=True)

    # Look for any .fna already present (handles multi-chromosome assemblies)
    existing = list(dest_dir.glob('*.fna'))
    if existing:
        return existing[0]

    print(f'  Downloading {gcf} for {host_id} ...')
    with tempfile.TemporaryDirectory() as tmp:
        zip_path = pathlib.Path(tmp) / 'dataset.zip'
        result = subprocess.run(
            ['datasets', 'download', 'genome', 'accession', gcf,
             '--include', 'genome', '--filename', str(zip_path)],
            capture_output=True, text=True
        )
        if result.returncode != 0:
            raise RuntimeError(f'datasets failed for {gcf}: {result.stderr}')

        with zipfile.ZipFile(zip_path) as zf:
            fna_names = [n for n in zf.namelist() if n.endswith('.fna')]
            if not fna_names:
                raise FileNotFoundError(f'No .fna in datasets zip for {gcf}')
            fna_path = dest_dir / f'{host_id}.fna'
            with zf.open(fna_names[0]) as src, open(fna_path, 'wb') as dst:
                dst.write(src.read())
    return fna_path

host_fna_paths = {}
for _, row in manifest.iterrows():
    try:
        fna = fetch_host_genome(row['host_id'], row['assembly'], HOST_GENOMES_DIR)
        host_fna_paths[row['host_id']] = fna
        print(f'  ✓ {row["host_id"]:45s} → {fna.name}')
    except Exception as e:
        print(f'  ✗ {row["host_id"]}: {e}')

print(f'\n{len(host_fna_paths)}/{len(manifest)} host genomes ready.')

In [ ]:
# ── Cell 4: Load phage DNA from pharokka .gbk files ─────────────────────────
# Concatenates all contigs per phage into a single sequence for GC computation.
# 拼接每個噬菌體的所有 contig，計算全基因組 GC%。

pairs = pd.read_csv(TRUTH_TABLE_PATH)
phage_ids = pairs['phage_id'].unique()
print(f'Truth table: {len(pairs)} rows, {len(phage_ids)} unique phages')

phage_records = {}  # phage_id -> {'seq': concatenated_str, 'length': int}
for phage_id in phage_ids:
    gbk_path = PHAGE_RUNS_DIR / phage_id / 'pharokka.gbk'
    if not gbk_path.exists():
        print(f'  ✗ Missing: {gbk_path}')
        continue
    contigs = list(SeqIO.parse(gbk_path, 'genbank'))
    concat_seq = ''.join(str(r.seq) for r in contigs)
    phage_records[phage_id] = {'seq': concat_seq, 'length': len(concat_seq)}
    print(f'  ✓ {phage_id:15s}  {len(contigs)} contig(s)  {len(concat_seq):,} bp')

print(f'\n{len(phage_records)}/{len(phage_ids)} phage genomes loaded.')

In [ ]:
# ── Cell 5: Compute per-genome GC% ──────────────────────────────────────────
# gc_fraction() returns a float 0-1 (e.g. 0.5312 = 53.12% GC).

gc_rows = []

# -- Phages --
print('Phage GC (vs pharokka pre-computed as sanity check):')
print(f'{"phage_id":20s}  {"gc_ours":>8s}  {"gc_pharokka":>11s}  {"delta":>7s}  {"flag":>5s}')
for phage_id, rec in phage_records.items():
    gc = gc_fraction(rec['seq'])
    gc_rows.append({'genome_id': phage_id, 'kind': 'phage',
                    'gc_whole': gc, 'length_bp': rec['length']})

    # Sanity check against pharokka
    tsv_path = PHAGE_RUNS_DIR / phage_id / 'pharokka_length_gc_cds_density.tsv'
    if tsv_path.exists():
        pharokka_gc = pd.read_csv(tsv_path, sep='\t')['gc_perc'].iloc[0]
        delta = abs(gc - pharokka_gc)
        flag = '⚠️' if delta > 0.005 else '✓'
        print(f'{phage_id:20s}  {gc:8.4f}  {pharokka_gc:11.4f}  {delta:7.4f}  {flag}')
    else:
        print(f'{phage_id:20s}  {gc:8.4f}  {"N/A":>11s}')

# -- Hosts --
print('\nHost GC:')
kind_map = manifest.set_index('host_id')['kind'].to_dict()
for host_id, fna_path in host_fna_paths.items():
    contigs = list(SeqIO.parse(fna_path, 'fasta'))
    concat_seq = ''.join(str(r.seq) for r in contigs)
    gc = gc_fraction(concat_seq)
    length = len(concat_seq)
    kind = kind_map.get(host_id, 'host')
    gc_rows.append({'genome_id': host_id, 'kind': kind,
                    'gc_whole': gc, 'length_bp': length})
    print(f'  {host_id:45s}  gc={gc:.4f}  ({length:,} bp)  [{kind}]')

gc_df = pd.DataFrame(gc_rows)
gc_csv = OUT_DIR / 'per_genome_gc.csv'
gc_df.to_csv(gc_csv, index=False)
print(f'\n✓ Saved: {gc_csv}')
display(gc_df)

In [ ]:
# ── Cell 6 & 7: Build per-pair feature + Z-score normalize ──────────────────
# x_gc = |GC_phage − GC_host|   (whole-genome fractions)

gc_lookup = gc_df.set_index('genome_id')['gc_whole'].to_dict()
kind_lookup = gc_df.set_index('genome_id')['kind'].to_dict()

feat_rows = []
for _, row in pairs.iterrows():
    p, h, y = row['phage_id'], row['host_id'], row['y']
    gc_p = gc_lookup.get(p)
    gc_h = gc_lookup.get(h)
    if gc_p is None or gc_h is None:
        print(f'  ✗ Missing GC for pair ({p}, {h}) — skipping')
        continue
    feat_rows.append({
        'phage_id':  p,
        'host_id':   h,
        'y':         y,
        'host_kind': kind_lookup.get(h, 'unknown'),
        'gc_phage':  gc_p,
        'gc_host':   gc_h,
        'x_gc':      abs(gc_p - gc_h),
    })

feat_df = pd.DataFrame(feat_rows)

# Z-score normalize x_gc
mu, sigma = feat_df['x_gc'].mean(), feat_df['x_gc'].std()
feat_df['x_gc_zscore'] = (feat_df['x_gc'] - mu) / sigma

feat_csv = OUT_DIR / 'f01_gc_content.csv'
feat_df.to_csv(feat_csv, index=False)
print(f'✓ Saved: {feat_csv}')
print(f'  {len(feat_df)} pairs  |  μ(x_gc)={mu:.4f}  σ={sigma:.4f}')
display(feat_df.head(10))

In [ ]:
# ── Cell 8: Exploration plots & correlation stats ────────────────────────────

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle('Factor 1 — GC Content Analysis', fontsize=14, fontweight='bold')

# ── Plot A: GC% distribution by genome group ────────────────────────────────
ax = axes[0]
group_colors = {'phage': '#E07B54', 'host': '#4C9A6E', 'neg': '#5B7EC9'}
group_labels = {'phage': 'Phage', 'host': 'Xanthomonas host', 'neg': 'Cross-genus neg'}
for kind, grp in gc_df.groupby('kind'):
    grp['gc_whole'].plot.kde(
        ax=ax, label=group_labels.get(kind, kind),
        color=group_colors.get(kind, 'grey'), lw=2
    )
    ax.axvline(grp['gc_whole'].mean(), color=group_colors.get(kind, 'grey'),
               linestyle='--', alpha=0.5, lw=1)
ax.set_xlabel('Whole-genome GC fraction')
ax.set_ylabel('Density')
ax.set_title('GC% distribution by group\nGC% 分布（依群組）')
ax.legend(fontsize=8)

# ── Plot B: Scatter x_gc vs y ────────────────────────────────────────────────
ax = axes[1]
for kind, grp in feat_df.groupby('host_kind'):
    jitter_y = grp['y'] + np.random.uniform(-0.05, 0.05, len(grp))
    ax.scatter(grp['x_gc'], jitter_y,
               label=group_labels.get(kind, kind),
               color=group_colors.get(kind, 'grey'),
               alpha=0.7, edgecolors='white', linewidths=0.5, s=60)
ax.set_xlabel('|GC_phage − GC_host|  (x_gc)')
ax.set_ylabel('y  (1 = infects, 0 = no)')
ax.set_title('x_gc vs infection label\nx_gc 與感染標籤')
ax.set_yticks([0, 1])
ax.legend(fontsize=8)

# ── Plot C: Box/strip of x_gc by y ───────────────────────────────────────────
ax = axes[2]
sns.boxplot(data=feat_df, x='y', y='x_gc', ax=ax,
            palette={0: '#5B7EC9', 1: '#4C9A6E'}, width=0.4)
sns.stripplot(data=feat_df, x='y', y='x_gc', ax=ax,
              color='black', alpha=0.4, size=5, jitter=True)
ax.set_xlabel('y label (0 = no infection, 1 = infects)')
ax.set_ylabel('|GC_phage − GC_host|')
ax.set_title('GC difference by outcome\n各感染結果之 GC 差異')

plt.tight_layout()
plot_path = OUT_DIR / 'gc_density.png'
fig.savefig(plot_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'✓ Saved: {plot_path}')

# ── Correlation stats ────────────────────────────────────────────────────────
pearson_r, pearson_p   = stats.pearsonr(feat_df['x_gc'], feat_df['y'])
spearman_r, spearman_p = stats.spearmanr(feat_df['x_gc'], feat_df['y'])

print('\n── Correlation: x_gc vs y ──────────────────────────────')
print(f'  Pearson  r={pearson_r:+.3f}  p={pearson_p:.3e}')
print(f'  Spearman r={spearman_r:+.3f}  p={spearman_p:.3e}')
print('  (Negative r expected: larger GC diff → less likely to infect)')

# ── Sanity table ─────────────────────────────────────────────────────────────
sanity = feat_df.groupby('y')['x_gc'].agg(['mean','std','count']).rename(
    index={0:'y=0 (no infection)', 1:'y=1 (infects)'})
print('\n── Sanity: mean x_gc by outcome ────────────────────────')
display(sanity)

## Results & interpretation / 結果解讀

**What to look for / 觀察重點:**

- **GC% distribution:** *Xanthomonas* hosts typically cluster around **63–66% GC**. Phages that co-evolved with them should fall near the same range. Cross-genus negative controls (e.g. *S. aureus* ~33%, *M. smegmatis* ~67%) widen the `x_gc` distribution, making negatives easier to separate.
- **宿主 GC 分布：** *黃單胞菌*宿主的 GC% 通常集中在 **63–66%**。與其共演化的噬菌體應落在相近範圍內。跨屬陰性對照（如金黃色葡萄球菌 ~33%，耻垢分枝桿菌 ~67%）會拉大 `x_gc` 分布，有助於區分陰性樣本。

- **Correlation sign:** Pearson / Spearman should be **negative** (higher `x_gc` → lower infection probability). A weak or absent signal here is not alarming — GC alone is a coarse proxy and all six factors are needed for the regression.
- **相關係數符號：** Pearson / Spearman 應為**負值**（`x_gc` 越大 → 感染機率越低）。若此單因子信號較弱，不需擔心——GC 含量只是粗略代理指標，迴歸模型需綜合六個因子才能發揮效用。

---

## Outputs written / 已寫出的輸出

| File | Description |
|------|-------------|
| `per_genome_gc.csv` | Per-genome GC fraction and length / 每個基因組的 GC 比例與長度 |
| `f01_gc_content.csv` | Per-pair feature `x_gc` and Z-score / 每對配對的 `x_gc` 特徵值與 Z 分數 |
| `gc_density.png` | Three-panel exploration plot / 三格探索圖 |

---

## Scaling to the full 777-genome pool / 擴展至 777 個噬菌體

Once pharokka is run on all 777 phage genomes (`02_annotation/processes/pharokka/`), extend by:
1. Adding the new phage accessions to `pairs.csv` (or replacing it with the updated truth table).
2. Re-running this notebook — Cell 4 will find the new `pharokka.gbk` files automatically.

No code changes required.

全部 777 個噬菌體完成 pharokka 註釋後，只需：
1. 將新登錄號加入 `pairs.csv`（或替換為更新版真值表）。
2. 重新執行本筆記本——Cell 4 將自動讀取新的 `pharokka.gbk` 檔案。無需修改程式碼。